# Churn Prediction Model v3 FINAL final
**Author:** Data Science Team  
**Last run:** Some Tuesday

> ⚠️ **NOTE:** Do not touch cells 3, 7, or 11. They were working.
> If something breaks, try restarting the kernel and running all.
> If it still breaks, ask Priya — she understands the preprocessing logic.

In [ ]:
# ── FAILURE 1: No reproducible environment ─────────────────────────
# There is no requirements.txt. No conda environment. No Docker image.
# This ran on Priya's laptop with xgboost 1.6. Good luck.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier          # might fail — version unknown
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')   # famous last words

In [ ]:
# ── Load data ──────────────────────────────────────────────────────
# The CSV is on Priya's Desktop. She emailed it to the team in February.
# If you don't have it, ask her (she might be on holiday).

df = pd.read_csv('data/raw/telco_churn.csv')
print(f'Loaded {len(df)} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# ── FAILURE 3: Silent data failure (TotalCharges dtype change) ──────
# TotalCharges is sometimes a string (spaces for missing values).
# We just drop NaN — what could go wrong?
# Answer: if the column is already a string, pd.to_numeric is not called,
# and ALL predictions silently default to the majority class.

df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)  
df.dropna(inplace=True)          # ← silently drops rows with TotalCharges = ' '
# No type conversion. No validation. No alert if shape changes drastically.

df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print(f'After cleanup: {len(df)} rows')

In [ ]:
# ── Encode categoricals ────────────────────────────────────────────
# This encoding logic is not documented anywhere.
# It assumes column order is stable across runs. It is not.

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('customerID')

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# ── Train model ────────────────────────────────────────────────────
# Hyperparameters chosen by gut feel. Not tracked anywhere.
# We tried 50+ combinations — none of them are documented.

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred))

In [ ]:
# ── FAILURE 5: No automated tests ──────────────────────────────────
# We 'test' by eyeballing the accuracy number.
# If we accidentally change the preprocessing above, this cell still runs.
# Accuracy could drop from 0.91 to 0.64 and nobody would know.

# Look at confusion matrix:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(cm)
# Looks reasonable? Ship it.

In [ ]:
# ── FAILURE 2: No model versioning ─────────────────────────────────
# Five model files exist in the models/ folder.
# We will save another one now. Nobody knows which one production uses.

import os
os.makedirs('models', exist_ok=True)

# We have: model_v1.pkl, model_v2.pkl, model_final.pkl,
#          model_final_v2.pkl, model_ACTUALLY_final.pkl
# and now...

with open('models/model_ACTUALLY_final.pkl', 'wb') as f:
    pickle.dump(model, f)

print('Model saved to models/model_ACTUALLY_final.pkl')
print('(There are now 6 model files. Production uses one of them.)')

In [ ]:
# ── FAILURE 4: Manual retraining reminder ──────────────────────────
# TODO: run this cell manually every Monday morning
# (This is a Jupyter notebook. There is no scheduler. No alert.
#  If you forget — and you WILL forget — the model silently ages.)

# Slack message from last month:
# @channel did anyone retrain the model this week?
# I was on holiday. Priya was at a conference. Rahul thought someone else did it.
# The model went 6 weeks without retraining.

print('Retraining complete! (If someone remembered to run this.)')

In [ ]:
# ── FAILURE 6: Manual deployment ───────────────────────────────────
# 'Deployment' = email the .pkl file to the engineering team
# and ask them to put it on the server. Manually. By hand.

# Email template:
# ---------------------------------------------------------------
# To: engineering@company.com
# Subject: New churn model - please deploy
#
# Hi team,
# Please find attached the new model file.
# Replace /srv/models/model.pkl with this file.
# Let me know if you need anything.
# Thanks, Data Science
# ---------------------------------------------------------------

# Issues with this 'deployment process':
# 1. No versioning — which email has the right model?
# 2. No rollback — can't go back to the previous model easily
# 3. No confirmation — did engineering actually replace the file?
# 4. No tests — did the new model break the API?
# 5. Human error — what if they put it in the wrong directory?

print('Please email this file to engineering@company.com')
print('Subject: New churn model attached, please deploy when you get a chance')

---
## Summary of problems with this notebook:

| # | Problem | Risk |
|---|---------|------|
| 1 | No environment spec | Breaks on any other machine |
| 2 | No model versioning | Cannot tell which model is in production |
| 3 | Silent data failure | All predictions wrong, no error raised |
| 4 | Manual retraining | Model ages; forgotten during holidays |
| 5 | No tests | Bad model deployed undetected |
| 6 | Manual deployment | Human error, no rollback, no audit trail |

**Every single one of these is fixed by the end of this workshop.**